<div style="background-color:#000047; padding: 30px; border-radius: 10px; color: white; text-align: center;">
    <img src='Figures/alinco_white_text.png' style="height: 100px; margin-bottom: 10px;"/>
    <h1>Aprendizaje Automático Avanzado</h1>
    <h3>Ejemplo: Gradient Boosting (XGBoost, LightGBM y CatBoost)</h3>
    
</div>

> En este ejemplo práctico implementamos un clasificador con **XGBoost** y, sobre el **mismo dataset**, entrenamos también **LightGBM** y **CatBoost**. Al final **comparamos** los tres modelos (exactitud, AUC y tiempo de entrenamiento) y concluimos **cuál es mejor y por qué**. El problema consiste en clasificar clientes de un mayorista en dos canales: **Horeca** (Hotel/Restaurante/Café) o **Retail** (comercio minorista).

**XGBoost** significa **Extreme Gradient Boosting**. Es un algoritmo de Machine Learning muy potente que domina el ML aplicado y las competencias de Kaggle. Es una implementación de **árboles con gradient boosting** diseñada para **velocidad y precisión**.

Tiene una **alta capacidad predictiva** y puede ser hasta ~10 veces más rápido que otras técnicas de boosting. Incluye varios **parámetros de regularización** que reducen el sobreajuste, por lo que también se le conoce como técnica de **boosting regularizado**.


### Gradient boosting
El **gradient boosting** combina las estimaciones de varios modelos **débiles** (árboles pequeños). Los árboles se construyen de forma **secuencial**: cada árbol nuevo intenta **reducir los errores** del anterior, dando más peso a los ejemplos mal clasificados. Así, cada árbol aprende de los **residuos** del previo.

### XGBoost
En XGBoost ajustamos cada modelo sobre el **gradiente de la función de pérdida** del paso anterior, generalizando el gradient boosting para que funcione con **cualquier función de pérdida diferenciable** y añadiendo **regularización**.

```mermaid
flowchart LR
    D[Datos] --> T1[Árbol 1]
    T1 --> E1[Residuos / errores]
    E1 --> T2[Árbol 2]
    T2 --> E2[Residuos / errores]
    E2 --> T3[Árbol 3]
    T3 --> S[Suma ponderada]
    S --> P[Predicción final]

    classDef root fill:#33ccff,color:#000,stroke:#000047;
    class D root;
```

## Planteamiento del problema

Resolvemos un problema de **clasificación binaria**: clasificar a los clientes de un mayorista en dos canales — **Horeca** (Hotel/Restaurante/Café) o **Retail** (minorista) — a partir de su gasto anual en distintas categorías de productos.

## 4. Descripción del dataset

Usamos el **Wholesale customers data set** del repositorio UCI. Contiene **440 registros** y **8 atributos** (todos numéricos): el canal, la región y el gasto anual en 6 categorías de productos.

| Variable | Descripción |
|---|---|
| `Channel` | **Objetivo:** canal (1 = Horeca, 2 = Retail) |
| `Region` | Región geográfica |
| `Fresh` | Gasto anual en productos frescos |
| `Milk` | Gasto anual en lácteos |
| `Grocery` | Gasto anual en abarrotes |
| `Frozen` | Gasto anual en congelados |
| `Detergents_Paper` | Gasto anual en detergentes y papel |
| `Delicassen` | Gasto anual en delicatessen |

> Fuente: https://archive.ics.uci.edu/ml/datasets/Wholesale+customers

In [ ]:
# Instalar las bibliotecas de boosting (ejecutar una vez; luego puedes comentar esta línea)
#%pip install -q xgboost lightgbm catboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

### Cargar el dataset

Cargamos los datos **directamente desde la UCI** (reproducible, sin archivos locales).

In [ ]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv'
df = pd.read_csv(url)

# Alternativa local: df = pd.read_csv('C:/datasets/Wholesale customers data.csv')
df.shape

Hay **440 instancias** y **8 atributos** en el dataset.

### Análisis exploratorio de datos

La variable `Channel` toma valores `1` (Horeca) y `2` (Retail): es nuestra variable objetivo.

In [ ]:
# No hay valores faltantes


### Definir variables predictoras y objetivo

Separamos las variables predictoras `X` de la objetivo `y`. Convertimos las etiquetas de `{1, 2}` a `{1, 0}`: **1 = Horeca**, **0 = Retail**.

### Dividir en entrenamiento y prueba

In [ ]:
from sklearn.model_selection import train_test_split



### Entrenar el clasificador **XGBoost**

XGBoost tiene muchos hiperparámetros. Los más importantes:

| Parámetro | Función |
|---|---|
| `learning_rate` | Tamaño de paso (contra el sobreajuste), rango [0, 1] |
| `max_depth` | Profundidad máxima de cada árbol |
| `subsample` | % de muestras usadas por árbol |
| `colsample_bytree` | % de variables usadas por árbol |
| `n_estimators` | Número de árboles |
| `objective` | Función de pérdida (`binary:logistic` para clasificación binaria) |
| `gamma`, `alpha` (L1), `lambda` (L2) | Regularización para reducir la complejidad |

In [ ]:
from xgboost import XGBClassifier



### Predicciones y exactitud

In [ ]:
from sklearn.metrics import accuracy_score


### Validación cruzada k-fold

Para obtener modelos más robustos usamos **validación cruzada k-fold** con la función `cv()` de XGBoost sobre una `DMatrix` (estructura optimizada de XGBoost). Medimos el **AUC**.

In [ ]:
import xgboost as xgb


### Importancia de características

XGBoost permite examinar qué variables usa más el modelo (número de veces que se divide sobre cada una).

> Normalmente la característica más importante en este dataset es `Grocery`. Así, XGBoost también sirve para **selección de características**.

### Modelo con **LightGBM**

**LightGBM** (Microsoft) es otra implementación de gradient boosting, optimizada para **velocidad** gracias al crecimiento *leaf-wise* y al uso de histogramas. Entrenamos sobre el **mismo dataset**.

In [ ]:
from lightgbm import LGBMClassifier


### Modelo con **CatBoost**

**CatBoost** (Yandex) destaca por su robustez y por manejar variables categóricas de forma nativa. Usa árboles **simétricos** y *ordered boosting* para reducir el sobreajuste. Entrenamos sobre el **mismo dataset**.

In [ ]:
from catboost import CatBoostClassifier


### Comparación de los tres modelos

Entrenamos los tres con configuraciones equivalentes y medimos **exactitud (accuracy)**, **AUC** y **tiempo de entrenamiento**.

In [ ]:
import time
from sklearn.metrics import roc_auc_score

modelos = {
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                             eval_metric='logloss', random_state=0),
    'LightGBM': LGBMClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=0),
    'CatBoost': CatBoostClassifier(iterations=100, depth=4, learning_rate=0.1,
                                   verbose=0, random_state=0),
}

def entrena_evalua(nombre, modelo):
    t0 = time.time()
    modelo.fit(X_train, y_train)
    dt = time.time() - t0
    proba = modelo.predict_proba(X_test)[:, 1]
    return {
        'Modelo': nombre,
        'Accuracy': accuracy_score(y_test, modelo.predict(X_test)),
        'AUC': roc_auc_score(y_test, proba),
        'Tiempo (s)': dt,
    }

tabla = pd.DataFrame([entrena_evalua(n, m) for n, m in modelos.items()])
tabla = tabla.sort_values('Accuracy', ascending=False).reset_index(drop=True)
tabla

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
colores = ['#2E9AFE', '#16a085', '#8e44ad']

ax[0].bar(tabla['Modelo'], tabla['Accuracy'], color=colores)
ax[0].set_ylim(0.8, 1.0)
ax[0].set_title('Accuracy en test')
for i, v in enumerate(tabla['Accuracy']):
    ax[0].text(i, v + 0.003, f'{v:.3f}', ha='center')

ax[1].bar(tabla['Modelo'], tabla['Tiempo (s)'], color=colores)
ax[1].set_title('Tiempo de entrenamiento (s)')
for i, v in enumerate(tabla['Tiempo (s)']):
    ax[1].text(i, v, f'{v:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

mejor = tabla.iloc[0]
print(f"Mejor modelo por accuracy: {mejor['Modelo']} "
      f"(accuracy={mejor['Accuracy']:.4f}, AUC={mejor['AUC']:.4f})")

### ¿cuál es mejor y por qué?

Sobre este dataset **pequeño** (440 registros, 7 variables numéricas), los tres algoritmos de gradient boosting alcanzan un rendimiento **muy similar** (accuracy ≈ 0.90–0.92). Las diferencias son pequeñas y pueden variar con la semilla, por lo que conviene mirar la **tabla comparativa** (accuracy + AUC + tiempo) de la sección anterior.

### Interpretación por modelo

| Modelo | Fortalezas en este caso | Cuándo brilla |
|---|---|---|
| **XGBoost** | Muy robusto y con fuerte regularización (`alpha`, `lambda`) | Estándar de la industria; datasets medianos |
| **LightGBM** | El **más rápido** de entrenar (histogramas + *leaf-wise*) | **Grandes volúmenes** de datos |
| **CatBoost** | Buenos resultados **sin apenas ajuste**; robusto al sobreajuste | Muchas **variables categóricas** / poco *tuning* |

### ¿Cuál elegir?

- **Para este problema (datos pequeños y numéricos):** el mejor es el que encabeza la tabla comparativa (según *accuracy*/AUC). Suelen quedar **CatBoost o XGBoost** ligeramente por delante, porque su regularización controla mejor el sobreajuste en datasets pequeños, donde el crecimiento *leaf-wise* de LightGBM puede sobreajustar un poco.
- **Si el criterio es la velocidad o el dataset es grande:** **LightGBM** es la mejor opción por su eficiencia, con una exactitud prácticamente equivalente.
- **Si hay muchas variables categóricas o se busca el mínimo esfuerzo de ajuste:** **CatBoost** es la opción más cómoda y robusta.

> **Conclusión:** no hay un ganador universal (*No Free Lunch*). Para este dataset tabular pequeño, **XGBoost/CatBoost** ofrecen la mejor exactitud por su regularización, mientras que **LightGBM** gana en velocidad. La recomendación práctica es **probar los tres** con validación cruzada y elegir según el equilibrio **precisión ↔ tiempo** que exija el proyecto.
